In [ ]:
import os
import numpy as np
import tensorflow as tf
import h5py
from tensorflow.keras import layers, Model

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
def load_paired_reynolds_multi_3channel(file_paths, lr_dim, hr_dim):
    """
    Loads data from multiple HDF5 files and stacks u, v, p as 3 channels.
    Data is kept on its native physical grid.
    """
    xs_lr_all, xs_hr_all, used_res_all, bc_types_all = [], [], [], []
    lx_all, ly_all = [], []

    for file_path in file_paths:
        print(f"\nLoading from: {file_path}")
        try:
            with h5py.File(file_path, "r") as f:
                all_keys = list(f.keys())
                if not all_keys:
                    print("  File is empty, skipping.")
                    continue

                re_numbers_in_file = sorted(list(set(int(k.split("_")[0][2:]) for k in all_keys if k.startswith("Re"))))
                print(f"  Found Reynolds numbers: {re_numbers_in_file}")

                first_group = f[all_keys[0]]
                bc_type = first_group.attrs.get("bc_type", "unknown")
                print(f"  BC Type: {bc_type}")

                for re_value in re_numbers_in_file:
                    g_lr = f"Re{re_value}_mesh{lr_dim}x{lr_dim}"
                    g_hr = f"Re{re_value}_mesh{hr_dim}x{hr_dim}"

                    if g_lr in all_keys and g_hr in all_keys:
                        if all(comp in f[g_lr] and comp in f[g_hr] for comp in ["u", "v", "p"]):
                            lr_u_raw = f[g_lr]["u"][()].astype(np.float32)
                            lr_v_raw = f[g_lr]["v"][()].astype(np.float32)
                            lr_p_raw = f[g_lr]["p"][()].astype(np.float32)

                            hr_u_raw = f[g_hr]["u"][()].astype(np.float32)
                            hr_v_raw = f[g_hr]["v"][()].astype(np.float32)
                            hr_p_raw = f[g_hr]["p"][()].astype(np.float32)

                            lx_lr = float(f[g_lr].attrs.get("lx", 1.0))
                            ly_lr = float(f[g_lr].attrs.get("ly", 1.0))
                            lx_hr = float(f[g_hr].attrs.get("lx", 1.0))
                            ly_hr = float(f[g_hr].attrs.get("ly", 1.0))

                            lr_u = lr_u_raw.reshape(lr_dim, lr_dim, order="F")
                            lr_v = lr_v_raw.reshape(lr_dim, lr_dim, order="F")
                            lr_p = lr_p_raw.reshape(lr_dim, lr_dim, order="F")

                            hr_u = hr_u_raw.reshape(hr_dim, hr_dim, order="F")
                            hr_v = hr_v_raw.reshape(hr_dim, hr_dim, order="F")
                            hr_p = hr_p_raw.reshape(hr_dim, hr_dim, order="F")

                            xs_lr_all.append(np.stack([lr_u, lr_v, lr_p], axis=-1))
                            xs_hr_all.append(np.stack([hr_u, hr_v, hr_p], axis=-1))
                            used_res_all.append(re_value)
                            bc_types_all.append(bc_type)
                            lx_all.append(lx_lr)
                            ly_all.append(ly_lr)
        except (IOError, OSError, FileNotFoundError) as exc:
            print(f"  Error opening file: {exc}")
            continue

    if len(xs_lr_all) == 0:
        raise ValueError("No data loaded from the provided H5 files.")

    print(f"\nTotal loaded: {len(xs_lr_all)} samples from {len(file_paths)} file(s)")
    print(f"BC Type distribution: {dict(zip(*np.unique(bc_types_all, return_counts=True)))}")

    return (
        np.array(xs_lr_all, dtype=np.float32),
        np.array(xs_hr_all, dtype=np.float32),
        np.array(used_res_all),
        np.array(bc_types_all),
        np.array(lx_all, dtype=np.float32),
        np.array(ly_all, dtype=np.float32),
    )


def bicubic_interpolate_batch(x, target_size):
    return tf.image.resize(x, target_size, method="bicubic")


def make_coord_channels_batch(dim, lx_arr, ly_arr):
    n_samples = len(lx_arr)
    coords = np.zeros((n_samples, dim, dim, 2), dtype=np.float32)
    col_idx = np.arange(dim, dtype=np.float32)
    row_idx = np.arange(dim, dtype=np.float32)
    for k in range(n_samples):
        scale = max(float(lx_arr[k]), float(ly_arr[k]))
        x_norm = col_idx / max(dim - 1, 1) * (float(lx_arr[k]) / scale)
        y_norm = row_idx / max(dim - 1, 1) * (float(ly_arr[k]) / scale)
        xx, yy = np.meshgrid(x_norm, y_norm)
        coords[k, :, :, 0] = xx
        coords[k, :, :, 1] = yy
    return coords


def normalize_per_sample(arr):
    n_samples = arr.shape[0]
    normalized = np.zeros_like(arr)
    stats = np.zeros((n_samples, 3, 2), dtype=np.float64)
    for i in range(n_samples):
        for ch in range(3):
            ch_data = arr[i, ..., ch]
            mean = np.mean(ch_data, dtype=np.float64)
            std = np.std(ch_data, dtype=np.float64)
            std = max(std, 1e-8)
            normalized[i, ..., ch] = (ch_data - mean) / std
            stats[i, ch, 0] = mean
            stats[i, ch, 1] = std
    return normalized.astype(np.float32), stats


def apply_per_sample_stats(arr, stats):
    normalized = np.zeros_like(arr)
    for i in range(arr.shape[0]):
        for ch in range(3):
            mean, std = stats[i, ch]
            normalized[i, ..., ch] = (arr[i, ..., ch] - mean) / std
    return normalized.astype(np.float32)


def denormalize_per_sample(arr, stats):
    denormalized = np.zeros_like(arr)
    for i in range(arr.shape[0]):
        for ch in range(3):
            mean, std = stats[i, ch]
            denormalized[i, ..., ch] = arr[i, ..., ch] * std + mean
    return denormalized.astype(np.float32)

In [ ]:
def gradient_difference_loss(y_true, y_pred):
    true_dx = y_true[:, :, 1:, :] - y_true[:, :, :-1, :]
    true_dy = y_true[:, 1:, :, :] - y_true[:, :-1, :, :]

    pred_dx = y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]
    pred_dy = y_pred[:, 1:, :, :] - y_pred[:, :-1, :, :]

    loss_x = tf.reduce_mean(tf.abs(true_dx - pred_dx))
    loss_y = tf.reduce_mean(tf.abs(true_dy - pred_dy))
    return loss_x + loss_y


def spectral_loss(y_true, y_pred):
    y_true_complex = tf.cast(y_true, tf.complex64)
    y_pred_complex = tf.cast(y_pred, tf.complex64)
    fft_true = tf.signal.fft2d(y_true_complex)
    fft_pred = tf.signal.fft2d(y_pred_complex)
    mag_true = tf.abs(fft_true)
    mag_pred = tf.abs(fft_pred)
    return tf.reduce_mean(tf.abs(mag_true - mag_pred))


class CompositeLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=1.0, beta=0.1, gamma=0.05, adaptive=False, name="composite_loss"):
        super().__init__(name=name)
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.adaptive = adaptive
        if adaptive:
            self.l1_ema = tf.Variable(1.0, trainable=False)
            self.grad_ema = tf.Variable(1.0, trainable=False)
            self.spec_ema = tf.Variable(1.0, trainable=False)
            self.ema_decay = 0.9

    def call(self, y_true, y_pred):
        l1_loss = tf.reduce_mean(tf.abs(y_true - y_pred))
        grad_loss = gradient_difference_loss(y_true, y_pred)
        spec_loss = spectral_loss(y_true, y_pred)

        if self.adaptive:
            self.l1_ema.assign(self.ema_decay * self.l1_ema + (1 - self.ema_decay) * l1_loss)
            self.grad_ema.assign(self.ema_decay * self.grad_ema + (1 - self.ema_decay) * grad_loss)
            self.spec_ema.assign(self.ema_decay * self.spec_ema + (1 - self.ema_decay) * spec_loss)

            alpha_adaptive = 1.0 / (self.l1_ema + 1e-8)
            beta_adaptive = 1.0 / (self.grad_ema + 1e-8)
            gamma_adaptive = 1.0 / (self.spec_ema + 1e-8)

            total = alpha_adaptive + beta_adaptive + gamma_adaptive
            alpha_adaptive /= total
            beta_adaptive /= total
            gamma_adaptive /= total

            total_loss = alpha_adaptive * l1_loss + beta_adaptive * grad_loss + gamma_adaptive * spec_loss
        else:
            total_loss = self.alpha * l1_loss + self.beta * grad_loss + self.gamma * spec_loss

        return total_loss


def conv_block(x, filters, kernel_size=3, activation="relu"):
    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x


def build_residual_unet(input_shape, base_filters=32, depth=3, activation="relu"):
    flow_channels = input_shape[-1] - 2
    inputs = layers.Input(shape=input_shape, name="unet_input")

    encoder_outputs = []
    x = inputs
    for i in range(depth):
        filters = base_filters * (2 ** i)
        x = conv_block(x, filters, activation=activation)
        x = conv_block(x, filters, activation=activation)
        encoder_outputs.append(x)
        if i < depth - 1:
            x = layers.MaxPooling2D(pool_size=2)(x)

    for i in range(depth - 2, -1, -1):
        filters = base_filters * (2 ** i)
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
        x = layers.Concatenate()([x, encoder_outputs[i]])
        x = conv_block(x, filters, activation=activation)
        x = conv_block(x, filters, activation=activation)

    residual = layers.Conv2D(flow_channels, 1, padding="same", name="residual_output")(x)
    return Model(inputs, residual, name="residual_unet")


class InterpolateRefineModel(Model):
    def __init__(self, unet_model, target_size, **kwargs):
        super().__init__(**kwargs)
        self.unet = unet_model
        self.target_size = target_size

    def call(self, inputs, training=False):
        interpolated = bicubic_interpolate_batch(inputs, self.target_size)
        residual = self.unet(interpolated, training=training)
        residual = residual * 0.1
        refined = interpolated[..., :3] + residual
        return refined

In [ ]:
# ========================= USER CONFIGURATION =========================
LR_DIM = 10
STAGE_DIMS = [20, 40, 80, 200, 400]

EPOCHS_PER_STAGE = [120, 120, 120, 120, 120]
BATCH_SIZE = 4
BASE_FILTERS = 32
UNET_DEPTH = 3
ACTIVATION = "swish"
LOSS_CONFIG = {"alpha": 1.0, "beta": 0.1, "gamma": 0.05, "adaptive": True}
FINE_TUNE_LR = 5e-5
TRAIN_SPLIT = 0.85
SHUFFLE_SEED = 42

BURGERS_H5 = r"/kaggle/input/datasets/amirmohdk/burgers-2d-file-for-fine-tuning/burgers2d_fields.h5"
FILE_PATHS = [BURGERS_H5]

PRETRAINED_MODELS = {
    "10to20": "/kaggle/input/datasets/amirmohdk/sr-unet-pretrained-model-upto-bfs-2-cases/unet_stage_10to20_progressive_residual_unet_(20-40-80-200-400)_trained along with bfs 100300.h5",
    "20to40": "/kaggle/input/datasets/amirmohdk/sr-unet-pretrained-model-upto-bfs-2-cases/unet_stage_20to40_progressive_residual_unet_(20-40-80-200-400)_trained along with bfs 100300.h5",
    "40to80": "/kaggle/input/datasets/amirmohdk/sr-unet-pretrained-model-upto-bfs-2-cases/unet_stage_40to80_progressive_residual_unet_(20-40-80-200-400)_trained along with bfs 100300.h5",
    "80to200": "/kaggle/input/datasets/amirmohdk/sr-unet-pretrained-model-upto-bfs-2-cases/unet_stage_80to200_progressive_residual_unet_(20-40-80-200-400)_trained along with bfs 100300.h5",
    "200to400": "/kaggle/input/datasets/amirmohdk/sr-unet-pretrained-model-upto-bfs-2-cases/unet_stage_200to400_progressive_residual_unet_(20-40-80-200-400)_trained along with bfs 100300.h5",
}

FINE_TUNE_TAG = "burgers2d_finetune"
SAVE_DIR = "outputs"

In [ ]:
print("\n" + "=" * 70)
print("LOADING BURGERS DATA FOR ALL STAGES")
print("=" * 70)

all_dims = [LR_DIM] + STAGE_DIMS
data_by_resolution = {}

for dim in all_dims:
    print(f"\nLoading {dim}x{dim}...")
    x_data, _, res_data, bc_data, lx_data, ly_data = load_paired_reynolds_multi_3channel(
        FILE_PATHS,
        lr_dim=dim,
        hr_dim=dim,
    )
    data_by_resolution[dim] = {
        "x": x_data,
        "res": res_data,
        "bc": bc_data,
        "lx": lx_data,
        "ly": ly_data,
    }
    print(f"  Loaded shape: {x_data.shape}")

n_samples = data_by_resolution[LR_DIM]["x"].shape[0]
if n_samples < 2:
    print("Warning: only one sample found; validation will reuse the same sample.")

indices = np.arange(n_samples)
rng = np.random.default_rng(SHUFFLE_SEED)
rng.shuffle(indices)

n_train = max(1, int(n_samples * TRAIN_SPLIT))
train_idx = indices[:n_train]
val_idx = indices[n_train:] if n_samples > 1 else indices[:1]

train_data = {}
val_data = {}
train_lx = {}
val_lx = {}
train_ly = {}
val_ly = {}

for dim in all_dims:
    train_data[dim] = data_by_resolution[dim]["x"][train_idx]
    val_data[dim] = data_by_resolution[dim]["x"][val_idx]
    train_lx[dim] = data_by_resolution[dim]["lx"][train_idx]
    val_lx[dim] = data_by_resolution[dim]["lx"][val_idx]
    train_ly[dim] = data_by_resolution[dim]["ly"][train_idx]
    val_ly[dim] = data_by_resolution[dim]["ly"][val_idx]

print(f"\nTrain samples: {train_data[LR_DIM].shape[0]}")
print(f"Val samples:   {val_data[LR_DIM].shape[0]}")

In [ ]:
print("\n" + "=" * 70)
print("APPLYING PER-SAMPLE NORMALIZATION AND COORD CHANNELS")
print("=" * 70)

train_data_norm = {}
val_data_norm = {}

train_data_norm[LR_DIM], train_stats = normalize_per_sample(train_data[LR_DIM])
val_data_norm[LR_DIM], val_stats = normalize_per_sample(val_data[LR_DIM])

for dim in STAGE_DIMS:
    train_data_norm[dim] = apply_per_sample_stats(train_data[dim], train_stats)
    val_data_norm[dim] = apply_per_sample_stats(val_data[dim], val_stats)

train_coords = {}
val_coords = {}
train_data_5ch = {}
val_data_5ch = {}

for dim in all_dims:
    train_coords[dim] = make_coord_channels_batch(dim, train_lx[dim], train_ly[dim])
    val_coords[dim] = make_coord_channels_batch(dim, val_lx[dim], val_ly[dim])
    train_data_5ch[dim] = np.concatenate([train_data_norm[dim], train_coords[dim]], axis=-1)
    val_data_5ch[dim] = np.concatenate([val_data_norm[dim], val_coords[dim]], axis=-1)

print("Normalization and coord channels ready.")

In [ ]:
def train_stage(model, x_input, x_target, epochs, batch_size, loss_config, stage_name, lr):
    print(f"\n{'='*60}")
    print(f"Fine-tuning Stage: {stage_name}")
    print(f"Input shape: {x_input.shape}, Target shape: {x_target.shape}")
    print(f"Loss config: {loss_config}")
    print(f"{'='*60}")

    n_samples = len(x_input)
    n_val = max(1, int(n_samples * 0.15))
    n_train = max(1, n_samples - n_val)

    indices = np.random.permutation(n_samples)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:] if n_samples > 1 else indices[:1]

    x_train = x_input[train_idx]
    y_train = x_target[train_idx]
    x_val = x_input[val_idx]
    y_val = x_target[val_idx]

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=CompositeLoss(**loss_config),
        jit_compile=False,
    )

    train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    train_dataset = train_dataset.shuffle(len(x_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=40,
        restore_best_weights=True,
        verbose=1,
        mode="min",
    )

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=[early_stopping],
        verbose=0,
    )

    best_epoch = np.argmin(history.history["val_loss"]) + 1
    best_val_loss = np.min(history.history["val_loss"])
    final_train_loss = history.history["loss"][-1]

    print(f"Stage {stage_name} fine-tune complete.")
    print(f"Best epoch: {best_epoch}/{len(history.history['loss'])}")
    print(f"Best val_loss: {best_val_loss:.6f}")
    print(f"Final train_loss: {final_train_loss:.6f}")

    return history


print("\n" + "=" * 70)
print("FINE-TUNING PRETRAINED CASCADED UNETS")
print("=" * 70)

trained_models = {}

x_train_input_prev = None
x_val_input_prev = None

for stage_idx, (target_dim, epochs) in enumerate(zip(STAGE_DIMS, EPOCHS_PER_STAGE)):
    input_dim = LR_DIM if stage_idx == 0 else STAGE_DIMS[stage_idx - 1]
    stage_name = f"{input_dim}to{target_dim}"

    model_file = PRETRAINED_MODELS.get(stage_name)
    if model_file is None:
        raise ValueError(f"Missing pretrained model for stage {stage_name}")
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Pretrained model not found: {model_file}")

    print(f"\nLoading pretrained U-Net: {model_file}")
    unet = tf.keras.models.load_model(model_file, compile=False)

    stage_model = InterpolateRefineModel(
        unet_model=unet,
        target_size=(target_dim, target_dim),
        name=f"stage_{stage_name}",
    )

    if stage_idx == 0:
        x_train_input = train_data_5ch[input_dim]
        x_val_input = val_data_5ch[input_dim]
    else:
        prev_model = trained_models[STAGE_DIMS[stage_idx - 1]]
        x_train_flow = prev_model.predict(x_train_input_prev, verbose=0)
        x_val_flow = prev_model.predict(x_val_input_prev, verbose=0)
        x_train_input = np.concatenate([x_train_flow, train_coords[input_dim]], axis=-1)
        x_val_input = np.concatenate([x_val_flow, val_coords[input_dim]], axis=-1)

    x_train_target = train_data_norm[target_dim]
    x_val_target = val_data_norm[target_dim]

    x_train_input_prev = x_train_input
    x_val_input_prev = x_val_input

    train_stage(
        model=stage_model,
        x_input=x_train_input,
        x_target=x_train_target,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        loss_config=LOSS_CONFIG,
        stage_name=stage_name,
        lr=FINE_TUNE_LR,
    )

    trained_models[target_dim] = stage_model

    save_name = f"unet_stage_{stage_name}_{FINE_TUNE_TAG}.h5"
    save_path = os.path.join(SAVE_DIR, save_name)
    stage_model.unet.save(save_path)
    print(f"Saved fine-tuned U-Net: {save_path}")

print("\nFine-tuning complete.")